In [1]:
# Graphiques
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss curve
ax1.plot(range(1, epochs+1), train_losses, marker='o', label='Train Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training Loss')
ax1.legend()
ax1.grid(alpha=0.3)

# Accuracy curve (100 - MAE)
ax2.plot(range(1, epochs+1), train_accs, marker='o', label='Train Accuracy', linewidth=2)
ax2.axhline(test_accs[0], color='red', linestyle='--', label=f'Test Accuracy: {test_accs[0]:.2f}%', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training Accuracy (100 - MAE)')
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print("\n✓ Graphiques affichés")

NameError: name 'plt' is not defined

In [ ]:
# Test
print("\n=== TESTING ===")
model.eval()
test_losses = []
test_accs = []

with torch.no_grad():
    total_loss = 0
    total_mae = 0
    n_batches = 0
    
    for images, ages in test_loader:
        images = images.to(device)
        ages = ages.to(device).long()
        
        logits, probs = model(images)
        loss = criterion(logits, ages)
        
        predicted_ages = calc_expected_age(probs)
        mae = torch.abs(predicted_ages - ages.float()).mean()
        
        total_loss += loss.item()
        total_mae += mae.item()
        n_batches += 1
    
    avg_loss = total_loss / n_batches
    avg_mae = total_mae / n_batches
    test_losses.append(avg_loss)
    test_accs.append(100 - avg_mae)
    
    print(f"Test Loss: {avg_loss:.4f}, Test MAE: {avg_mae:.2f} years")
    print(f"Test Accuracy (~): {100 - avg_mae:.2f}%")

In [ ]:
# Entraînement
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 10
train_losses = []
train_accs = []

print("\n=== TRAINING ===")
for epoch in range(epochs):
    model.train()
    total_loss = 0
    total_mae = 0
    n_batches = 0
    
    for images, ages in train_loader:
        images = images.to(device)
        ages = ages.to(device).long()
        
        # Forward
        logits, probs = model(images)
        loss = criterion(logits, ages)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Metrics
        predicted_ages = calc_expected_age(probs)
        mae = torch.abs(predicted_ages - ages.float()).mean()
        
        total_loss += loss.item()
        total_mae += mae.item()
        n_batches += 1
    
    avg_loss = total_loss / n_batches
    avg_mae = total_mae / n_batches
    train_losses.append(avg_loss)
    train_accs.append(100 - avg_mae)  # Accuracy ~= 100 - MAE
    
    print(f"Epoch {epoch+1}/{epochs} - Loss: {avg_loss:.4f}, MAE: {avg_mae:.2f} years")

In [ ]:
# Données synthétiques pour test
class SimpleAgeDataset(Dataset):
    def __init__(self, n_samples=1000):
        self.images = torch.randn(n_samples, 3, 227, 227)
        self.ages = torch.randint(0, 101, (n_samples,)).float()
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        return self.images[idx], self.ages[idx]

# Créer train/test datasets
train_dataset = SimpleAgeDataset(800)
test_dataset = SimpleAgeDataset(200)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"✓ Dataset créé - Train: {len(train_dataset)}, Test: {len(test_dataset)}")

In [ ]:
# Modèle DEX: prédire probabilité pour 101 âges puis weighted average
class DEXModel(nn.Module):
    def __init__(self, num_classes=101):
        super(DEXModel, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=11, stride=4, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(64, 192, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        logits = self.classifier(x)
        probs = torch.softmax(logits, dim=1)
        return logits, probs

model = DEXModel(AGE_BINS).to(device)
print(f"✓ Modèle DEX créé")

# Expected value from probabilities
def calc_expected_age(probs):
    age_range = torch.arange(0, AGE_BINS, dtype=torch.float32, device=device)
    return torch.sum(probs * age_range.unsqueeze(0), dim=1)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

AGE_BINS = 101  # Ages 0-100

# DEX (Deep EXpectation) - Age Estimation